# **Семинар 1.** Введение в NLP. Токенизация. Векторные представления текстов. Классификация текстов

В машинном обучении мы обычно имеем дело с матрицами «объекты — признаки», где каждый объект представлен вектором фиксированной размерности $x_i \in \mathbb{R}^d$. Кроме того, модели машинного обучения умеют работать только с числовыми признаками. На практике же часто приходится работать и с другими видами данных, например с текстами.

Сегодня мы поговорим о том, как представить текст в числовом виде, пригодном для подачи на вход моделям машинного обучения.

## 1.1. Токенизация

Для начала разделим текст на составные части, с которыми будет работать модель. Введем несколько определений:

* **Токен** — единица текста, с которой работает модель.

* **Токенизация** — процесс разбиения текста на последовательность токенов.

* **Токенизатор** — алгоритм токенизации, который разбивает текст на токены по некоторому набору правил.

Существует большое количество библиотек для обработки текстовых данных в Python. Мы начнем с Natural Language Toolkit (NLTK). NLTK содержит множество встроенных токенизаторов.

In [5]:
from nltk import tokenize

[element for element in dir(tokenize) if element[0].isupper()]

['BlanklineTokenizer',
 'LegalitySyllableTokenizer',
 'LineTokenizer',
 'MWETokenizer',
 'NLTKWordTokenizer',
 'PunktSentenceTokenizer',
 'PunktTokenizer',
 'RegexpTokenizer',
 'ReppTokenizer',
 'SExprTokenizer',
 'SpaceTokenizer',
 'StanfordSegmenter',
 'SyllableTokenizer',
 'TabTokenizer',
 'TextTilingTokenizer',
 'ToktokTokenizer',
 'TreebankWordDetokenizer',
 'TreebankWordTokenizer',
 'TweetTokenizer',
 'WhitespaceTokenizer',
 'WordPunctTokenizer']

In [6]:
from nltk.tokenize import RegexpTokenizer, SpaceTokenizer, TweetTokenizer, WhitespaceTokenizer, WordPunctTokenizer

In [14]:
text = """
Добро пожаловать на курс по NLP! :)

Давайте токенизируем наш первый текст. В нем есть слова, пунктуация и даже... хэштеги?

#HSE #FBB #MachineLearning #NaturalLanguageProcessing
"""

tokenizers = [
    RegexpTokenizer(r"\w+"),
    RegexpTokenizer(r"\s+", gaps=True),
    SpaceTokenizer(),
    TweetTokenizer(),
    WhitespaceTokenizer(),
    WordPunctTokenizer(),
]

for tokenizer in tokenizers:
    print(tokenizer.__class__.__name__)
    print(tokenizer.tokenize(text))
    print()

RegexpTokenizer
['Добро', 'пожаловать', 'на', 'курс', 'по', 'NLP', 'Давайте', 'токенизируем', 'наш', 'первый', 'текст', 'В', 'нем', 'есть', 'слова', 'пунктуация', 'и', 'даже', 'хэштеги', 'HSE', 'FBB', 'MachineLearning', 'NaturalLanguageProcessing']

RegexpTokenizer
['Добро', 'пожаловать', 'на', 'курс', 'по', 'NLP!', ':)', 'Давайте', 'токенизируем', 'наш', 'первый', 'текст.', 'В', 'нем', 'есть', 'слова,', 'пунктуация', 'и', 'даже...', 'хэштеги?', '#HSE', '#FBB', '#MachineLearning', '#NaturalLanguageProcessing']

SpaceTokenizer
['\nДобро', 'пожаловать', 'на', 'курс', 'по', 'NLP!', ':)\n\nДавайте', 'токенизируем', 'наш', 'первый', 'текст.', 'В', 'нем', 'есть', 'слова,', 'пунктуация', 'и', 'даже...', 'хэштеги?\n\n#HSE', '#FBB', '#MachineLearning', '#NaturalLanguageProcessing\n']

TweetTokenizer
['Добро', 'пожаловать', 'на', 'курс', 'по', 'NLP', '!', ':)', 'Давайте', 'токенизируем', 'наш', 'первый', 'текст', '.', 'В', 'нем', 'есть', 'слова', ',', 'пунктуация', 'и', 'даже', '...', 'хэштеги',

## 1.2. Bag-of-Words

Теперь мы умеем делить тексты на токены. Но в одном тексте может быть 20 токенов, а в другом — десятки тысяч. Как превратить каждый из них в вектор фиксированной размерности?

Начнем с одного из самых простых подходов — **Bag-of-Words**, или «мешка слов». Пусть нам дан словарь $V = (v_1, \ldots, v_M)$, содержащий $M$ уникальных токенов. Тогда каждый текст мы можем представить в виде $M$-мерного вектора $x$:

$$
x_j = \sum_{k=1}^{L} \mathbf{1}[t_k = v_j],
$$

где $t_k$ — $k$-й токен данного текста, а $L$ — его длина. Иными словами, $x_j$ — это количество раз, которое токен $v_j$ встретился в тексте.

Но откуда же нам взять словарь $V$? Его можно построить на обучающей подвыборке, собрав все уникальные токены и присвоив каждому из них свой индекс. Разумеется, после построения словаря нам могут встретиться новые токены, которых в нем нет. Их будем называть **out-of-vocabulary (OOV)** и просто игнорировать.

In [15]:
from sklearn.feature_extraction.text import CountVectorizer

In [21]:
texts = [
    "Тук-тук-тук, Пенни! Тук-тук-тук, Пенни! Тук-тук-тук, Пенни!",
    "Эх, Марио, если б я только мог управлять людьми, как я управляю тобой...",
    "Я не сумасшедший. Моя мама меня проверяла.",
    "Вот ты где, нейтрино — мой маленький субатомный чертёнок!",
]

In [22]:
vectorizer = CountVectorizer(
    lowercase=False,
    tokenizer=WordPunctTokenizer().tokenize,
    token_pattern=None,
)
vectorizer.fit(texts)

,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",False
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",<bound method...NE|re.DOTALL)>
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",None
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word n-gram or charactern-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21Since v0.21, if ``input`` is ``filename`` or ``file``, the data isfirst read from the file and then passed to the given callableanalyzer.",'word'


In [26]:
bow = vectorizer.transform(texts)
bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 39 stored elements and shape (4, 36)>

In [27]:
import pandas as pd

bow = pd.DataFrame(bow.toarray(), columns=vectorizer.get_feature_names_out())
bow[sorted(bow.columns, key=lambda s: s.lower())]

,!,",",-,.,...,б,Вот,где,если,как,...,Тук,тук,ты,управлять,управляю,чертёнок,Эх,Я,я,—
0,3,3,6,0,0,0,0,0,0,0,...,3,6,0,0,0,0,0,0,0,0
1,0,3,0,0,1,1,0,0,1,1,...,0,0,0,1,1,0,1,0,2,0
2,0,0,0,2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,1,1,0,0,0,0,1,1,0,0,...,0,0,1,0,0,1,0,0,0,1


In [29]:
vectorizer.set_params(lowercase=True)

bow = vectorizer.fit_transform(texts)
bow = pd.DataFrame(bow.toarray(), columns=vectorizer.get_feature_names_out())
bow[sorted(bow.columns, key=lambda s: s.lower())]

,!,",",-,.,...,б,вот,где,если,как,...,тобой,только,тук,ты,управлять,управляю,чертёнок,эх,я,—
0,3,3,6,0,0,0,0,0,0,0,...,0,0,9,0,0,0,0,0,0,0
1,0,3,0,0,1,1,0,0,1,1,...,1,1,0,0,1,1,0,1,2,0
2,0,0,0,2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,1,1,0,0,0,0,1,1,0,0,...,0,0,0,1,0,0,1,0,0,1


С регистром разобрались. Но наш словарь по-прежнему содержит несколько форм одного и того же слова.

In [30]:
bow[["управлять", "управляю"]]

,управлять,управляю
0,0,0
1,1,1
2,0,0
3,0,0


In [32]:
bow[["мой", "моя"]]

,мой,моя
0,0,0
1,0,0
2,0,1
3,1,0


## 1.3. Стемминг и лемматизация

Нам нужен способ нормализации слов. Например, слова «мой» и «моя» можно объединить. Рассмотрим два подхода:

* **Стемминг** — удаление частей слова (аффиксов) по некоторому набору правил для данного языка. Результат может не являться корректным словом.

* **Лемматизация** — приведение слова к нормальной форме. Например, для существительных это именительный падеж и единственное число.

In [35]:
from nltk.stem import SnowballStemmer

stemmer = SnowballStemmer("russian")
stemmer.stem("стали")

'стал'

In [34]:
from pymorphy3 import MorphAnalyzer

morph = MorphAnalyzer()
morph.parse("стали")

[Parse(word='стали', tag=OpencorporaTag('VERB,perf,intr plur,past,indc'), normal_form='стать', score=0.975342, methods_stack=((DictionaryAnalyzer(), 'стали', 945, 4),)),
 Parse(word='стали', tag=OpencorporaTag('NOUN,inan,femn sing,gent'), normal_form='сталь', score=0.010958, methods_stack=((DictionaryAnalyzer(), 'стали', 13, 1),)),
 Parse(word='стали', tag=OpencorporaTag('NOUN,inan,femn plur,nomn'), normal_form='сталь', score=0.005479, methods_stack=((DictionaryAnalyzer(), 'стали', 13, 6),)),
 Parse(word='стали', tag=OpencorporaTag('NOUN,inan,femn sing,datv'), normal_form='сталь', score=0.002739, methods_stack=((DictionaryAnalyzer(), 'стали', 13, 2),)),
 Parse(word='стали', tag=OpencorporaTag('NOUN,inan,femn sing,loct'), normal_form='сталь', score=0.002739, methods_stack=((DictionaryAnalyzer(), 'стали', 13, 5),)),
 Parse(word='стали', tag=OpencorporaTag('NOUN,inan,femn plur,accs'), normal_form='сталь', score=0.002739, methods_stack=((DictionaryAnalyzer(), 'стали', 13, 9),))]

In [36]:
words = [
    "люди", "людьми", "человек",
    "твоего", "твоё", "твою",
    "бежать", "бежал", "бежала",
]  # fmt: skip

pd.DataFrame(
    {
        "Исходное слово": words,
        "Стемминг": [stemmer.stem(word) for word in words],
        "Лемматизация": [morph.parse(word)[0].normal_form for word in words],
    }
)

,Исходное слово,Стемминг,Лемматизация
0,люди,люд,человек
1,людьми,людьм,человек
2,человек,человек,человек
3,твоего,тво,твой
4,твоё,тво,твой
5,твою,тво,твой
6,бежать,бежа,бежать
7,бежал,бежа,бежать
8,бежала,бежа,бежать


## 1.4. Проблемы подхода Bag-of-Words

Одна из основных проблем BoW — большой словарь и неустойчивость даже к незначительным изменениям слов. Каждая форма слова, каждый вариант его написания порождает новый токен.

In [41]:
vectorizer.transform(["чертенок"]).toarray()

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

Кроме того, BoW — это векторное представление текста, которое не учитывает смысл отдельных слов.

In [53]:
words = ["врач", "доктор", "яблоко"]

bow = CountVectorizer().fit_transform(["врач", "доктор", "яблоко"])
bow.toarray()

array([[1, 0, 0],
       [0, 1, 0],
       [0, 0, 1]])

In [54]:
from sklearn.metrics.pairwise import cosine_distances

pd.DataFrame(cosine_distances(bow), index=words, columns=words)

,врач,доктор,яблоко
врач,0.0,1.0,1.0
доктор,1.0,0.0,1.0
яблоко,1.0,1.0,0.0


«Врач» одинаково далек от «доктора» и «яблока». Общий смысл не сделал их векторы ближе. Эту проблему пока оставим — к ней вернёмся на следующих семинарах.

Есть и другая проблема: Пенни и Эми могут поменяться ролями, хотя набор токенов в предложении останется тем же.

In [182]:
texts = ["Пенни обидела Эми", "Эми обидела Пенни"]

vectorizer = CountVectorizer()
bow = vectorizer.fit_transform(texts)

pd.DataFrame(bow.toarray(), index=texts, columns=vectorizer.get_feature_names_out())

,обидела,пенни,эми
Пенни обидела Эми,1,1,1
Эми обидела Пенни,1,1,1


Мешок слов не учитывает порядок слов. Обидно, но и с этим мы пока ничего сделать не можем. Давайте попробуем улучшить то, что он уже умеет: работать с частотами.

## 1.5. Пунктуация и стоп-слова

Рассмотрим три отзыва. Насколько похожими окажутся их векторы?

In [61]:
reviews = [
    "wonderful movie with brilliant acting and beautiful music.",
    "the movie was wonderful and it was one of the best films I have seen!",
    "the movie was terrible and it was one of the worst films I have seen!",
]

vectorizer = CountVectorizer(tokenizer=WordPunctTokenizer().tokenize, token_pattern=None)

bow = vectorizer.fit_transform(reviews)
pd.DataFrame(bow.toarray(), columns=vectorizer.get_feature_names_out())

,!,.,acting,and,beautiful,best,brilliant,films,have,i,...,music,of,one,seen,terrible,the,was,with,wonderful,worst
0,0,1,1,1,1,0,1,0,0,0,...,1,0,0,0,0,0,0,1,1,0
1,1,0,0,1,0,1,0,1,1,1,...,0,1,1,1,0,2,2,0,1,0
2,1,0,0,1,0,0,0,1,1,1,...,0,1,1,1,1,2,2,0,0,1


In [62]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_similarity(bow).round(3)

array([[1.   , 0.224, 0.149],
       [0.224, 1.   , 0.9  ],
       [0.149, 0.9  , 1.   ]])

Заметим, что пунктуация, а также слова *the*, *was*, *it* и *of* почти ничего не говорят о тональности отзыва. Однако они являются полноценными токенами и вносят свой вклад в сближение векторов второго и третьего текстов.

Такие часто встречающиеся служебные слова, которые обычно несут мало полезной информации для решения задачи, называют **стоп-словами (stop words)**.

In [64]:
from string import punctuation

punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [65]:
from nltk.corpus import stopwords

EN_STOPWORDS = stopwords.words("english")
EN_STOPWORDS

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

Их можно удалить на этапе предобработки текста.

In [66]:
def tokenize(text: str) -> list[str]:
    tokens = WordPunctTokenizer().tokenize(text.lower())
    return [t for t in tokens if t not in punctuation and t not in EN_STOPWORDS]


tokenize("the movie was terrible and it was one of the worst films I have seen!")

['movie', 'terrible', 'one', 'worst', 'films', 'seen']

In [67]:
vectorizer = CountVectorizer(tokenizer=tokenize, token_pattern=None)

bow = vectorizer.fit_transform(reviews)
bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 18 stored elements and shape (3, 12)>

In [68]:
cosine_similarity(bow).round(3)

array([[1.   , 0.333, 0.167],
       [0.333, 1.   , 0.667],
       [0.167, 0.667, 1.   ]])

Видно, что удаление пунктуации и стоп-слов немного исправило ситуацию, однако делать это нужно с осторожностью. Так, например, отрицания также входят в список стоп-слов, что может сыграть с нами злую шутку при решении задачи анализа тональности текста (sentiment analysis). А некоторые знаки препинания могут использоваться для выражения эмоций (например, скобки).

In [70]:
RU_STOPWORDS = stopwords.words("russian")
"не" in RU_STOPWORDS

True

In [71]:
reviews = [
    "фильм мне не понравился",
    "фильм мне совсем не понравился",
    "фильм мне понравился",
]

vectorizer = CountVectorizer(tokenizer=WordPunctTokenizer().tokenize, stop_words=RU_STOPWORDS, token_pattern=None)

bow = vectorizer.fit_transform(reviews)
pd.DataFrame(bow.toarray(), columns=vectorizer.get_feature_names_out())

,понравился,фильм
0,1,1
1,1,1
2,1,1


In [72]:
cosine_similarity(bow).round(3)

array([[1., 1., 1.],
       [1., 1., 1.],
       [1., 1., 1.]])

## 1.6. TF–IDF

Заметим также, что слово «movie» встречается во всех трех отзывах. Оно вносит свой вклад в Bag-of-Words и, как следствие, в cosine similarity, но почти не несет полезной информации.

Для удаления слишком частых слов при построении BoW мы можем воспользоваться параметром `max_df`. Однако есть и более интересный способ: не удалять такие слова совсем, а уменьшать их вклад.

Идея TF-IDF (Term Frequency – Inverse Document Frequency) проста: вес токена должен зависеть от того, как часто он встречается в данном документе и насколько редко — в остальных.

Пусть $N$ — число документов, $M$ — размер словаря, а $x_{ij}$ — число вхождений токена $v_j$ в документ $d_i$.

$$
\operatorname{TF}_{ij} = x_{ij},
\qquad
\operatorname{DF}_j = \sum_{i=1}^{N} \mathbf{1}[x_{ij} > 0],
$$

где $\operatorname{TF}_{ij}$ — частота токена в документе, а $\operatorname{DF}_j$ — число документов, в которых встретился токен $v_j$. Повторения внутри одного документа $\operatorname{DF}_j$ не увеличивают.

Теперь определим inverse document frequency:

$$
\operatorname{IDF}_j
=
\log\frac{N}{\operatorname{DF}_j}
\qquad
\left(
\operatorname{IDF}_j
=
\log\frac{N+1}{\operatorname{DF}_j+1}+1
\right).
$$

Чем в большем числе документов встречается токен, тем меньше его IDF. Для редких токенов, наоборот, IDF будет больше.

Наконец, умножим TF на IDF:

$$
z_{ij} = \operatorname{TF}_{ij}\operatorname{IDF}_j.
$$

В `sklearn` полученный вектор по умолчанию дополнительно нормализуется до единичной длины:

$$
\hat{z}_{ij}
=
\frac{z_{ij}}
{\sqrt{\sum_{m=1}^{M} z_{im}^{2}}}.
$$


In [88]:
from sklearn.feature_extraction.text import TfidfVectorizer

reviews = [
    "wonderful movie with brilliant acting and beautiful music.",
    "the movie was wonderful and it was one of the best films I have seen!",
    "the movie was terrible and it was one of the worst films I have seen!",
]

background = ["... the movie ... it was one of ..."] * 42

reviews += background

vectorizer = TfidfVectorizer(tokenizer=tokenize, token_pattern=None)

tfidf = vectorizer.fit_transform(reviews)
pd.DataFrame(tfidf.toarray()[:3].round(3), columns=vectorizer.get_feature_names_out())

,...,acting,beautiful,best,brilliant,films,movie,music,one,seen,terrible,wonderful,worst
0,0.0,0.453,0.453,0.00,0.453,0.000,0.110,0.453,0.000,0.000,0.000,0.409,0.000
1,0.0,0.000,0.000,0.53,0.000,0.478,0.128,0.000,0.131,0.478,0.000,0.478,0.000
2,0.0,0.000,0.000,0.00,0.000,0.466,0.125,0.000,0.128,0.466,0.517,0.000,0.517


In [89]:
cosine_similarity(tfidf).round(3)[:3, :3]

array([[1.   , 0.209, 0.014],
       [0.209, 1.   , 0.478],
       [0.014, 0.478, 1.   ]])

## 1.7. Классификация текстов. Naive Bayes classifier

Одни слова чаще встречаются в восторженных отзывах, другие — в разгромных. Попробуем использовать это наблюдение для классификации. Рассмотрим BoW-представление текста $x = (x_1, \ldots, x_M)$, где $x_j$ — количество вхождений токена $v_j \in V$ в текст. Мы хотим выбрать для данного текста наиболее вероятный класс:

$$
\hat y = \arg\max_y P(y \mid x).
$$

По теореме Байеса:

$$
P(y \mid x)
=
\frac{P(x \mid y)P(y)}{P(x)}.
$$

Нас интересует только числитель этого выражения, присмотримся к нему повнимательнее. Оценить $P(y)$ несложно — это просто доля соответствующего класса в выборке.

Чтобы получить $P(x \mid y)$, сделаем предположение, что все токены в тексте *независимы* (в реальности это, разумеется, не так, поэтому предположение *наивное*, отсюда и название модели).

Обозначим

$$
\theta_{yj} = P(v_j \mid y)
$$

— вероятность встретить токен $v_j$ в документах класса $y$. Тогда

$$
P(x \mid y)
\propto
\prod_{j=1}^{M} \theta_{yj}^{x_j}.
$$

Если токен встретился в тексте $x_j$ раз, его вероятность входит в произведение $x_j$ раз.

Подставим это выражение:

$$
\hat y
=
\arg\max_y
P(y)
\prod_{j=1}^{M} \theta_{yj}^{x_j}.
$$

Чтобы не перемножать большое количество маленьких вероятностей, перейдем к логарифмам. Логарифм — монотонная функция, поэтому положение максимума не изменится:

$$
\boxed{
\hat y
=
\arg\max_y
\left[
\log P(y)
+
\sum_{j=1}^{M} x_j \log \theta_{yj}
\right]
}
$$

## 1.8. Анализ тональности текстов. Кинопоиск

От коротких реплик перейдем к [настоящим отзывам о фильмах](https://huggingface.co/datasets/ai-forever/kinopoisk-sentiment-classification). Их авторы могут хвалить фильм, ругать его или давать нейтральную оценку. Попробуем различить эти три случая по тексту.

In [91]:
from typing import Literal


def load_kinopoisk_reviews(split: Literal["train", "validation", "test"]) -> pd.DataFrame:
    return pd.read_json(
        f"https://huggingface.co/datasets/ai-forever/kinopoisk-sentiment-classification/resolve/main/{split}.jsonl",
        lines=True,
    )


train = load_kinopoisk_reviews("train")
train.sample(10, random_state=42)

,id,text,label,label_text
5118,5118,С первого момента фильм поражает своей бестолк...,0,Bad
8931,8931,"Да, мир твой, Тони Монтана. Ты твердо знал, че...",2,Good
8515,8515,Давайте представим такую вот вполне вероятную ...,0,Bad
7282,7282,"Знаете, знаете что я заметил? Никто не паникуе...",2,Good
7623,7623,"«Этот человек не Шерлок Холмс» — первая мысль,...",0,Bad
6029,6029,Когда я начал смотреть самый знаменитый фильм ...,1,Neutral
3329,3329,Никаких междометий типа «Вау» или «Ух ты» посл...,1,Neutral
8934,8934,Вчера посмотрел второй раз. На диске. Купил ег...,1,Neutral
3879,3879,Молодой львёнок Симба — сын короля. Он должен ...,2,Good
3781,3781,"Последовав за приятелем, начал оскароносный пр...",0,Bad


In [92]:
train["label"].value_counts()

label
2    3500
0    3500
1    3500
Name: count, dtype: int64

In [93]:
validation = load_kinopoisk_reviews("validation")
test = load_kinopoisk_reviews("test")

In [98]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline, make_pipeline


class KinopoiskTokenizer:
    def __init__(self) -> None:
        self._tokenizer = WordPunctTokenizer()
        self._stopwords = set(stopwords.words("russian")) - {"не"}
        self._stemmer = SnowballStemmer("russian")

    def tokenize(self, text: str) -> list[str]:
        tokens = self._tokenizer.tokenize(text.lower())

        return [self._stemmer.stem(t) for t in tokens if t.isalpha() and t not in self._stopwords]


vectorizer_params = {
    "tokenizer": KinopoiskTokenizer().tokenize,
    "token_pattern": None,
    "max_df": 0.9,
    "min_df": 5,
    "max_features": 50_000,
}

pipelines: dict[str, Pipeline] = {
    "BoW + Naive Bayes": make_pipeline(
        CountVectorizer(**vectorizer_params),
        MultinomialNB(),
    ),
    "BoW + Logistic Regression": make_pipeline(
        CountVectorizer(**vectorizer_params),
        LogisticRegression(max_iter=1000),
    ),
    "TF-IDF + Logistic Regression": make_pipeline(
        TfidfVectorizer(**vectorizer_params),
        LogisticRegression(max_iter=1000),
    ),
}

scores = {}

for name, pipeline in pipelines.items():
    pipeline.fit(train["text"], train["label"])

    preds = pipeline.predict(validation["text"])
    scores[name] = f1_score(validation["label"], preds, average="macro")

pd.Series(scores, name="Validation macro-F1").round(3)

BoW + Naive Bayes               0.625
BoW + Logistic Regression       0.641
TF-IDF + Logistic Regression    0.680
Name: Validation macro-F1, dtype: float64

Качество модели — не единственный критерий. Чем больше словарь, тем больше размер признакового пространства и тем дороже хранить представление текстов.

В отдельном отзыве встречается лишь небольшая часть слов из словаря, поэтому матрица Bag-of-Words содержит очень много нулей. Хранить такую матрицу целиком невыгодно. На практике используют разреженное представление: сохраняют только ненулевые значения и их позиции.

In [101]:
vectorizer = pipelines["BoW + Naive Bayes"][0]
bow = vectorizer.transform(train["text"])

n_documents, n_features = bow.shape
n_elements = n_documents * n_features

sparse_bytes = bow.data.nbytes + bow.indices.nbytes + bow.indptr.nbytes
dense_bytes = n_elements * bow.dtype.itemsize

print(f"Количество признаков: {n_features:,}")
print(f"Количество документов: {n_documents:,}")
print(f"Доля нулевых элементов: {1 - bow.nnz / n_elements:.2%}")
print(f"Размер разреженной матрицы: {sparse_bytes / 2**20:.1f} MiB")
print(f"Размер плотной матрицы: {dense_bytes / 2**20:.1f} MiB")

Количество признаков: 18,728
Количество документов: 10,500
Доля нулевых элементов: 99.27%
Размер разреженной матрицы: 16.5 MiB
Размер плотной матрицы: 1500.3 MiB


## 1.9. N-граммы

Но даже огромный словарь не помог бы понять, кто кого обидел в истории Пенни и Эми. До сих пор мы рассматривали каждое слово отдельно, не учитывая его окружение.

Попробуем сохранить немного контекста — соседние слова. Для этого будем использовать **словесные n-граммы**.

In [102]:
texts = ["Пенни обидела Эми", "Эми обидела Пенни"]

vectorizer = CountVectorizer(ngram_range=(1, 2))

bow = vectorizer.fit_transform(texts)
pd.DataFrame(bow.toarray(), index=texts, columns=vectorizer.get_feature_names_out())

,обидела,обидела пенни,обидела эми,пенни,пенни обидела,эми,эми обидела
Пенни обидела Эми,1,0,1,1,1,1,0
Эми обидела Пенни,1,1,0,1,0,1,1


In [103]:
texts = [
    "Шелдон не сумасшедший, он гений",
    "Шелдон не гений, он сумасшедший",
]

# vectorizer = CountVectorizer()
vectorizer = CountVectorizer(ngram_range=(1, 2))

bow = vectorizer.fit_transform(texts)
pd.DataFrame(bow.toarray(), index=texts, columns=vectorizer.get_feature_names_out())

,гений,гений он,не,не гений,не сумасшедший,он,он гений,он сумасшедший,сумасшедший,сумасшедший он,шелдон,шелдон не
"Шелдон не сумасшедший, он гений",1,0,1,0,1,1,1,0,1,1,1,1
"Шелдон не гений, он сумасшедший",1,1,1,1,0,1,0,1,1,0,1,1


Но достаточно даже небольшой опечатки, чтобы знакомое слово превратилось для модели в совершенно новый признак.

Попробуем разбивать слова на короткие фрагменты. Часть из них сохранится и при опечатке, и при изменении окончания. Так мы получим **символьные n-граммы**.

In [107]:
vectorizer = CountVectorizer(analyzer="char", ngram_range=(3, 3))
get_ngrams = vectorizer.build_analyzer()

In [108]:
for word in ("Bazinga!", "Bazzinga!"):
    print(get_ngrams(word))

['baz', 'azi', 'zin', 'ing', 'nga', 'ga!']
['baz', 'azz', 'zzi', 'zin', 'ing', 'nga', 'ga!']


In [109]:
for word in ("управлять", "управляю"):
    print(get_ngrams(word))

['упр', 'пра', 'рав', 'авл', 'вля', 'лят', 'ять']
['упр', 'пра', 'рав', 'авл', 'вля', 'ляю']


In [111]:
ngram_pipelines: dict[str, Pipeline] = {
    "BoW (word n-grams) + MultinomialNB": make_pipeline(
        CountVectorizer(ngram_range=(1, 2), **vectorizer_params),
        MultinomialNB(),
    ),
    "TF-IDF (char n-grams) + LogisticRegression": make_pipeline(
        TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 5),
            min_df=5,
            max_df=0.9,
            max_features=50_000,
        ),
        LogisticRegression(max_iter=1000),
    ),
}

scores = {}

for name, pipeline in ngram_pipelines.items():
    pipeline.fit(train["text"], train["label"])

    preds = pipeline.predict(validation["text"])
    scores[name] = f1_score(validation["label"], preds, average="macro")

pd.Series(scores, name="Validation macro-F1").round(3)

BoW (word n-grams) + MultinomialNB            0.636
TF-IDF (char n-grams) + LogisticRegression    0.688
Name: Validation macro-F1, dtype: float64

## 2.0. Интерпретация

Заглянем внутрь Naive Bayes classifier: какие признаки склоняют модель к положительной или отрицательной оценке? Посчитаем для токенов

$$
\log P(v_j \mid y=\text{Good}) - \log P(v_j \mid y=\text{Bad}).
$$

Чем больше это значение, тем сильнее токен связан с положительным классом; чем меньше — с отрицательным.

In [129]:
words = [
    "прекрасн",
    "отличн",
    "невероятн",
    "плох",
    "скучн",
    "ужасн",
    "сюжет",
    "актер",
    "сцен",
]

(_, vectorizer), (_, classifier) = pipelines["BoW + Naive Bayes"].steps

weights = pd.Series(
    classifier.feature_log_prob_[2] - classifier.feature_log_prob_[0],
    index=vectorizer.get_feature_names_out(),
)
weights.loc[words].sort_values().round(2)

ужасн       -1.25
скучн       -1.10
плох        -1.05
сюжет       -0.34
сцен        -0.33
актер       -0.11
невероятн    0.89
отличн       1.07
прекрасн     1.20
dtype: float64

In [130]:
from sklearn.metrics import classification_report

best_pipeline = ngram_pipelines["TF-IDF (char n-grams) + LogisticRegression"]
test_preds = best_pipeline.predict(test["text"])

print(classification_report(test["label"], test_preds, digits=3, zero_division=0))

              precision    recall  f1-score   support

           0      0.736     0.796     0.765       500
           1      0.556     0.480     0.515       500
           2      0.712     0.750     0.730       500

    accuracy                          0.675      1500
   macro avg      0.668     0.675     0.670      1500
weighted avg      0.668     0.675     0.670      1500

